# Solar Filament Segmentation 2026: run 3 report

Summarizes the output of `kaggle_run3.ipynb` without any GPU work: validation Dice of each
fold model, the tuned post-processing, and out-of-fold PQ for the threshold rule and the
LightGBM picker, on all 707 training images and on the 106 validation images of runs 1 and 2
(so the runs are compared on the same held-out images).

**Before running:** add the run 3 notebook's output as an input (*Add Input > Your Work >
Notebooks*) and turn Internet on. No accelerator is needed.

In [ ]:
REPO_URL = "https://github.com/FarnooshMemari/solar-filament-segmentation.git"
BRANCH = "run3"  # switch to "main" once the run 3 pull request is merged
REPO = "/tmp/solar-filament-segmentation"

import glob, json, os
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} {REPO_URL} {REPO}
%cd {REPO}
!git log -1 --format="code version: %h %s"
!pip install -q pycocotools

found = glob.glob("/kaggle/input/**/runs/run3/oof_pieces.npz", recursive=True)
assert found, "Add the run 3 notebook's output as an input (Add Input > Your Work > Notebooks)."
RUN3 = os.path.dirname(found[0])
RUNS = os.path.dirname(RUN3)
print("run 3 output:", RUNS)

## 1. Fold models

In [ ]:
for path in sorted(glob.glob(f"{RUNS}/fold*/metrics.jsonl")):
    rows = [json.loads(l) for l in open(path)]
    val = [r for r in rows if "val_dice" in r]
    best = max(val, key=lambda r: r["val_dice"])
    split = json.load(open(os.path.join(os.path.dirname(path), "split.json")))
    print(f"{os.path.basename(os.path.dirname(path))}: {len(split['train'])} train / {len(split['val'])} held-out images, "
          f"best validation Dice {best['val_dice']:.4f} at epoch {best['epoch']}, "
          f"last {val[-1]['val_dice']:.4f}, {sum(r['seconds'] for r in rows) / 60:.0f} min")

## 2. Tuned post-processing and the picker

In [ ]:
post = json.load(open(f"{RUN3}/postprocess.json"))
print("post-processing:", json.dumps(post, indent=1))
print("\ntop settings:")
for row in json.load(open(f"{RUN3}/tuning_results.json"))[:5]:
    print(f"  PQ {row['pq']:.4f}  {row['setting']}")
picker = json.load(open(f"{RUN3}/picker.json"))
print("\npicker cut-off:", picker["tau"], "| pieces:", picker["pieces"], "| features:", ", ".join(picker["features"]))
print("out-of-fold PQ:", {k: v["pq"] for k, v in picker["oof"].items()})
print("feature importance:", list(picker["feature_importance"].items())[:8])

## 3. Out-of-fold PQ on all images, per fold and on the run 1/2 validation images

In [ ]:
!python scripts/oof_report.py --run {RUN3} --out /kaggle/working/oof_report.json